In [1]:
# Verificar o status da GPU (se aplicável)
!nvidia-smi

Wed Jun  3 00:47:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Garantir que estamos no diretório base e que não existe um yolov5 anterior
%cd /content
!rm -rf yolov5  # Remove qualquer clone anterior

# Clonar o repositório YOLOv5
!git clone https://github.com/ultralytics/yolov5
%cd yolov5

# Instalar dependências do YOLOv5
%pip install -qr requirements.txt
%pip install -q roboflow

/content
Cloning into 'yolov5'...
remote: Enumerating objects: 17972, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 17972 (delta 91), reused 31 (delta 31), pack-reused 17856 (from 3)
Receiving objects: 100% (17972/17972), 17.10 MiB | 27.03 MiB/s, done.
Resolving deltas: 100% (12226/12226), done.
/content/yolov5


In [3]:
from google.colab import drive
import os

# Montar o Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Desmontar o Google Drive
# drive.flush_and_unmount()

In [4]:
import shutil
import yaml

# 1. Caminhos
caminho_zip_drive = "/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/Yolo_dataset.zip"
pasta_local_vm = "/content/Yolo_dataset"

# 2. Criar a pasta "caixinha" para não espalhar os arquivos
os.makedirs(pasta_local_vm, exist_ok=True)

# 3. Automação MLOps (Verifica se o YAML já está lá dentro)
if not os.path.exists(f"{pasta_local_vm}/data.yaml"):
    print("🚚 1. Copiando o ZIP do Drive para o SSD da VM...")
    shutil.copy2(caminho_zip_drive, "/content/Yolo_dataset.zip")
    
    print("📦 2. Descompactando na velocidade da luz...")
    # AGORA SIM! Descompactando para DENTRO da pasta_local_vm
    shutil.unpack_archive("/content/Yolo_dataset.zip", pasta_local_vm)
    
    print("🔧 3. Ajustando o data.yaml para o caminho do SSD local...")
    yaml_path = f"{pasta_local_vm}/data.yaml"
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    data['path'] = pasta_local_vm
    
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f, sort_keys=False)
        
    print("🚀 SUCESSO! Ambiente efêmero montado e pronto para treinar.")
else:
    print("✅ O dataset já está no SSD desta sessão e pronto para uso!")

# 4. Aponta a variável global do YOLO para o SSD local
os.environ["DATASET_DIRECTORY"] = pasta_local_vm

🚚 1. Copiando o ZIP do Drive para o SSD da VM...
📦 2. Descompactando na velocidade da luz...
🔧 3. Ajustando o data.yaml para o caminho do SSD local...
🚀 SUCESSO! Ambiente efêmero montado e pronto para treinar.


In [7]:
import torch

print(f"Setup completo. Usando torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

# Destrói qualquer arquivo de cache corrompido do YOLO nas pastas
!rm -f "$DATASET_DIRECTORY/labels/train/*.cache"
!rm -f "$DATASET_DIRECTORY/labels/val/*.cache"
!rm -f "$DATASET_DIRECTORY/images/train/*.cache"
!rm -f "$DATASET_DIRECTORY/images/val/*.cache"

# Desativa o Weights & Biases para evitar o prompt interativo no notebook
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
print("W&B desativado para este notebook.")

# Executa o treinamento com a variável de ambiente
# !python train.py --batch 16 --epochs 1 --data "$DATASET_DIRECTORY/data.yaml" --weights yolov5s.pt --cache
!python train.py --batch 64 --epochs 5 --data "$DATASET_DIRECTORY/data.yaml" --weights yolov5s.pt --workers 4 --cache disk


Setup completo. Usando torch 2.11.0+cu128 (Tesla T4)
W&B desativado para este notebook.
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
train: weights=yolov5s.pt, cfg=, data=/content/Yolo_dataset/data.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=5, batch_size=64, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=data/hyps, resume_evolve=None, bucket=, cache=disk, image_weights=False, device=, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=4, project=runs/train, name=exp, exist_ok=False, quad=False, cos_lr=False, label_smoothing=0.0, patience=100, freeze=[0], save_period=-1, seed=0, local_rank=-1, entity=None, upload_dataset=False, bbox_interval=-1, artifact_alias=latest, ndjson_console=False, ndjson_file=False
github: up to date with https://github.com/ultralytics/

In [9]:
import shutil

# Caminho de origem (onde o YOLO salvou)
origem = "/content/yolov5/runs/train/exp3"

# Caminho de destino (O seu Google Drive)
# Mudei o final para criar uma pasta "Resultados_YOLO"
destino = "/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/Resultados_YOLOv5"

# Copia a pasta inteira
shutil.copytree(origem, destino)

print("✅ Todos os gráficos e o modelo 'best.pt' foram salvos no seu Drive com sucesso!")

✅ Todos os gráficos e o modelo 'best.pt' foram salvos no seu Drive com sucesso!


In [ ]:
# Start tensorboard
# Launch after you have started training
# logs save in the folder "runs"
# %load_ext tensorboard
# %tensorboard --logdir runs